# Model A — Question & Answer Generator / Verifier
**Course:** Artificial Intelligence — BS(CS) Spring 2026  
**Institution:** NUCES FAST, Islamabad  
**Notebook:** 2 of 5 — Model A Training

---
### What this notebook covers
| Sub-task | Approach |
|---|---|
| Answer Verification | Logistic Regression, SVM, Naive Bayes, Random Forest, XGBoost |
| Unsupervised Clustering | K-Means, Gaussian Mixture Model |
| Semi-Supervised | Label Propagation |
| Question Generation | Template-based + SVM/RF ranker |
| Ensemble | Soft-vote + Stacking |

**Prerequisite:** Run `1_preprocessing.ipynb` first so that `/kaggle/working/processed/` exists.

## 0. Imports & Config

In [1]:
import os, re, warnings, joblib, time
import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.semi_supervised import LabelPropagation, LabelSpreading
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import normalize
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)

PROC  = '/kaggle/input/datasets/muhammadibrahim304/race-processed-dataset'
MOUT  = '/kaggle/working/models/model_a'
os.makedirs(MOUT, exist_ok=True)
print('Model output dir:', MOUT)

Model output dir: /kaggle/working/models/model_a


## 1. Load Preprocessed Data

In [2]:
print("Loading OHE feature matrices ...")
X_train = sp.load_npz(f'{PROC}/X_train_ohe.npz')
X_val   = sp.load_npz(f'{PROC}/X_val_ohe.npz')
X_test  = sp.load_npz(f'{PROC}/X_test_ohe.npz')

y_train = np.load(f'{PROC}/y_train_ohe.npy')
y_val   = np.load(f'{PROC}/y_val_ohe.npy')
y_test  = np.load(f'{PROC}/y_test_ohe.npy')

print(f"X_train: {X_train.shape}, y_train balance: {np.bincount(y_train)}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

# Load processed DataFrames (for multi-class accuracy calculation)
train_df = pd.read_csv(f'{PROC}/train_processed.csv')
val_df   = pd.read_csv(f'{PROC}/val_processed.csv')
test_df  = pd.read_csv(f'{PROC}/test_processed.csv')
le = joblib.load(f'{PROC}/label_encoder.pkl')
print("Data loaded.")

Loading OHE feature matrices ...
X_train: (245936, 20000), y_train balance: [184452  61484]
X_val:   (35136, 20000)
X_test:  (70284, 20000)
Data loaded.


## 2. Helper: Evaluation & Answer-Selection
The binary OHE matrices have 4 rows per original sample (one per option).  
We evaluate in two ways:
- **Binary** accuracy: is this option correctly labeled correct/incorrect?
- **Multi-class** accuracy: among the 4 options, does the model pick the right one?

In [3]:
def evaluate_binary(name, model, X, y, split='val'):
    t0 = time.time()
    y_pred = model.predict(X)
    elapsed = time.time() - t0
    acc  = accuracy_score(y, y_pred)
    f1   = f1_score(y, y_pred, average='macro')
    prec = precision_score(y, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y, y_pred, average='macro', zero_division=0)
    print(f"[{name}] {split} | Acc={acc:.4f}  Macro-F1={f1:.4f}  "
          f"Prec={prec:.4f}  Rec={rec:.4f}  ({elapsed:.1f}s)")
    return {'model': name, 'split': split, 'accuracy': acc,
            'macro_f1': f1, 'precision': prec, 'recall': rec}

def multiclass_accuracy(model, X_block, n_orig):
    """
    For each original sample (4 consecutive rows in X_block),
    pick option with highest predicted probability/score.
    Returns array of predicted option indices (0=A,1=B,2=C,3=D).
    """
    if hasattr(model, 'predict_proba'):
        probs = model.predict_proba(X_block)[:, 1]  # prob of being correct
    elif hasattr(model, 'decision_function'):
        probs = model.decision_function(X_block)
    else:
        probs = model.predict(X_block).astype(float)
    probs = probs.reshape(n_orig, 4)
    return np.argmax(probs, axis=1)

def mc_accuracy(model, X, true_labels_mc):
    """true_labels_mc: array of int 0-3 (correct option index) per sample."""
    n = len(true_labels_mc)
    preds = multiclass_accuracy(model, X, n)
    return accuracy_score(true_labels_mc, preds)

# True multi-class labels for val and test
y_val_mc  = le.transform(val_df['answer'])
y_test_mc = le.transform(test_df['answer'])

results = []  # collect all metric rows

## 3. Model A-1 — Logistic Regression

In [6]:
print("Training Logistic Regression ...")
lr = LogisticRegression(
    C=1.0,
    max_iter=500,
    solver='lbfgs',
    n_jobs=-1,
    random_state=42
)
t0 = time.time()
lr.fit(X_train, y_train)
print(f"  Training time: {time.time()-t0:.1f}s")

r = evaluate_binary('LR', lr, X_val, y_val)
r['mc_accuracy'] = mc_accuracy(lr, X_val, y_val_mc)
print(f"  Multi-class accuracy (val): {r['mc_accuracy']:.4f}")
results.append(r)
joblib.dump(lr, f'{MOUT}/lr_model.pkl')
print("  Saved.")

Training Logistic Regression ...
  Training time: 48.7s
[LR] val | Acc=0.7499  Macro-F1=0.4288  Prec=0.5417  Rec=0.5000  (0.0s)
  Multi-class accuracy (val): 0.3011
  Saved.


## 4. Model A-2 — Support Vector Machine (LinearSVC)

In [8]:
print("Training LinearSVC (Primal) ...")
# Added dual=False. You can also safely lower max_iter.
svm = LinearSVC(C=0.5, max_iter=1000, dual=False, random_state=42)

t0 = time.time()
svm.fit(X_train, y_train)
print(f"  Training time: {time.time()-t0:.1f}s")

r = evaluate_binary('SVM', svm, X_val, y_val)
r['mc_accuracy'] = mc_accuracy(svm, X_val, y_val_mc)
print(f"  Multi-class accuracy (val): {r['mc_accuracy']:.4f}")
results.append(r)
joblib.dump(svm, f'{MOUT}/svm_model.pkl')
print("  Saved.")

Training LinearSVC (Primal) ...
  Training time: 306.8s
[SVM] val | Acc=0.7478  Macro-F1=0.4331  Prec=0.5146  Rec=0.5004  (0.0s)
  Multi-class accuracy (val): 0.2970
  Saved.


## 5. Model A-3 — Complement Naive Bayes

In [9]:
print("Training Complement Naive Bayes ...")
# ComplementNB works well with sparse binary/count features
nb = ComplementNB(alpha=0.5)
t0 = time.time()
nb.fit(X_train, y_train)
print(f"  Training time: {time.time()-t0:.1f}s")

r = evaluate_binary('CNB', nb, X_val, y_val)
r['mc_accuracy'] = mc_accuracy(nb, X_val, y_val_mc)
print(f"  Multi-class accuracy (val): {r['mc_accuracy']:.4f}")
results.append(r)
joblib.dump(nb, f'{MOUT}/nb_model.pkl')
print("  Saved.")

Training Complement Naive Bayes ...
  Training time: 0.2s
[CNB] val | Acc=0.5369  Macro-F1=0.4858  Prec=0.5043  Rec=0.5057  (0.0s)
  Multi-class accuracy (val): 0.3366
  Saved.


## 6. Model A-4 — Random Forest (on dense lexical features)

In [10]:
# RF works better on dense features; use cosine + lexical features
cos_train_df = pd.read_csv(f'{PROC}/cos_features_train.csv')
cos_val_df   = pd.read_csv(f'{PROC}/cos_features_val.csv')
cos_test_df  = pd.read_csv(f'{PROC}/cos_features_test.csv')

# Repeat each cosine row 4 times (to match the 4-option expanded labels)
cos_train_rep = np.repeat(cos_train_df.values, 4, axis=0)
cos_val_rep   = np.repeat(cos_val_df.values,   4, axis=0)

# Combine with first 500 OHE columns (to keep RF fast)
X_train_dense = np.hstack([X_train[:, :500].toarray(), cos_train_rep])
X_val_dense   = np.hstack([X_val[:, :500].toarray(),   cos_val_rep])

print("Training Random Forest ...")
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    n_jobs=-1,
    random_state=42
)
t0 = time.time()
rf.fit(X_train_dense, y_train)
print(f"  Training time: {time.time()-t0:.1f}s")

r = evaluate_binary('RF', rf, X_val_dense, y_val)
r['mc_accuracy'] = mc_accuracy(rf, X_val_dense, y_val_mc)
print(f"  Multi-class accuracy (val): {r['mc_accuracy']:.4f}")
results.append(r)
joblib.dump(rf, f'{MOUT}/rf_model.pkl')
print("  Saved.")

Training Random Forest ...
  Training time: 62.0s
[RF] val | Acc=0.7500  Macro-F1=0.4286  Prec=0.3750  Rec=0.5000  (0.5s)
  Multi-class accuracy (val): 0.2237
  Saved.


## 7. Model A-5 — XGBoost

In [11]:
print("Training XGBoost ...")
xgb_clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1,
    random_state=42,
    tree_method='hist'
)
t0 = time.time()
xgb_clf.fit(
    X_train_dense, y_train,
    eval_set=[(X_val_dense, y_val)],
    verbose=50
)
print(f"  Training time: {time.time()-t0:.1f}s")

r = evaluate_binary('XGB', xgb_clf, X_val_dense, y_val)
r['mc_accuracy'] = mc_accuracy(xgb_clf, X_val_dense, y_val_mc)
print(f"  Multi-class accuracy (val): {r['mc_accuracy']:.4f}")
results.append(r)
xgb_clf.save_model(f'{MOUT}/xgb_model.json')
print("  Saved.")

Training XGBoost ...
[0]	validation_0-logloss:0.56233
[50]	validation_0-logloss:0.56234
[100]	validation_0-logloss:0.56235
[150]	validation_0-logloss:0.56236
[200]	validation_0-logloss:0.56236
[250]	validation_0-logloss:0.56237
[299]	validation_0-logloss:0.56237
  Training time: 25.7s
[XGB] val | Acc=0.7500  Macro-F1=0.4286  Prec=0.3750  Rec=0.5000  (0.1s)
  Multi-class accuracy (val): 0.2223
  Saved.


## 8. Unsupervised: K-Means Clustering

Group question-answer pairs by OHE feature similarity to discover  
latent answer patterns without using any labels.

In [12]:
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score

# Reduce dimensionality for clustering (K-Means on sparse is slow)
print("Reducing OHE features with TruncatedSVD (50 components) ...")
svd = TruncatedSVD(n_components=50, random_state=42)

# Use a sample for speed
SAMPLE = min(20_000, X_train.shape[0])
idx = np.random.choice(X_train.shape[0], SAMPLE, replace=False)
X_sample = svd.fit_transform(X_train[idx])
y_sample = y_train[idx]

print(f"  SVD explained variance ratio sum: {svd.explained_variance_ratio_.sum():.3f}")
joblib.dump(svd, f'{MOUT}/svd_50.pkl')

# K-Means with k=2 (correct / incorrect)
print("Fitting K-Means (k=2) ...")
km = KMeans(n_clusters=2, n_init=10, random_state=42)
km_labels = km.fit_predict(X_sample)

sil = silhouette_score(X_sample, km_labels, sample_size=5000)
print(f"  Silhouette score (k=2): {sil:.4f}")

# Purity (assign cluster to majority label)
from scipy.stats import mode as scipy_mode
def purity_score(true, pred):
    total = 0
    for c in np.unique(pred):
        mask = pred == c
        majority = scipy_mode(true[mask], keepdims=True).count[0]
        total += majority
    return total / len(true)

pur = purity_score(y_sample, km_labels)
print(f"  Clustering purity: {pur:.4f}")

# Also try k=4 (one cluster per option label)
km4 = KMeans(n_clusters=4, n_init=10, random_state=42)
km4_labels = km4.fit_predict(X_sample)
sil4 = silhouette_score(X_sample, km4_labels, sample_size=5000)
print(f"  Silhouette score (k=4): {sil4:.4f}")

joblib.dump(km, f'{MOUT}/kmeans_k2.pkl')
joblib.dump(km4, f'{MOUT}/kmeans_k4.pkl')

Reducing OHE features with TruncatedSVD (50 components) ...
  SVD explained variance ratio sum: 0.156
Fitting K-Means (k=2) ...
  Silhouette score (k=2): 0.0829
  Clustering purity: 0.7543
  Silhouette score (k=4): 0.0670


['/kaggle/working/models/model_a/kmeans_k4.pkl']

## 9. Unsupervised: Gaussian Mixture Model

Probabilistic soft-membership clustering — better for overlapping distributions.

In [13]:
print("Fitting Gaussian Mixture Model (n=2) ...")
gmm = GaussianMixture(n_components=2, covariance_type='full',
                      random_state=42, max_iter=200)
gmm.fit(X_sample)
gmm_labels = gmm.predict(X_sample)

sil_gmm = silhouette_score(X_sample, gmm_labels, sample_size=5000)
pur_gmm = purity_score(y_sample, gmm_labels)
print(f"  GMM Silhouette: {sil_gmm:.4f}")
print(f"  GMM Purity    : {pur_gmm:.4f}")

joblib.dump(gmm, f'{MOUT}/gmm_model.pkl')
print("  Saved.")

Fitting Gaussian Mixture Model (n=2) ...
  GMM Silhouette: 0.0691
  GMM Purity    : 0.7543
  Saved.


## 10. Semi-Supervised: Label Propagation

Use a small labeled set; propagate labels to nearby unlabeled samples  
in SVD-reduced feature space.

In [14]:
# Semi-supervised: label 20% of data, treat rest as unlabeled (-1)
LABELED_FRAC = 0.20
SS_SAMPLE = min(10_000, X_train.shape[0])   # keep small for memory

idx_ss = np.random.choice(X_train.shape[0], SS_SAMPLE, replace=False)
X_ss = svd.transform(X_train[idx_ss])       # use already-fit SVD
y_ss_true = y_train[idx_ss]

# Mask unlabeled
y_ss = y_ss_true.copy().astype(int)
n_unlabeled = int(SS_SAMPLE * (1 - LABELED_FRAC))
unlabeled_idx = np.random.choice(SS_SAMPLE, n_unlabeled, replace=False)
y_ss[unlabeled_idx] = -1

print(f"Label Propagation: {SS_SAMPLE - n_unlabeled} labeled, {n_unlabeled} unlabeled")
lp = LabelSpreading(kernel='knn', n_neighbors=7, max_iter=30)
lp.fit(X_ss, y_ss)

# Evaluate only on the originally unlabeled portion
y_lp_pred = lp.transduction_[unlabeled_idx]
y_lp_true = y_ss_true[unlabeled_idx]
lp_acc = accuracy_score(y_lp_true, y_lp_pred)
lp_f1  = f1_score(y_lp_true, y_lp_pred, average='macro')
print(f"  Label Propagation (unlabeled set) Acc={lp_acc:.4f}  Macro-F1={lp_f1:.4f}")

joblib.dump(lp, f'{MOUT}/label_prop_model.pkl')

Label Propagation: 2000 labeled, 8000 unlabeled
  Label Propagation (unlabeled set) Acc=0.6452  Macro-F1=0.5045


['/kaggle/working/models/model_a/label_prop_model.pkl']

## 11. Clustering vs. Supervised Comparison Table

In [15]:
unsup_table = pd.DataFrame([
    {'Method': 'K-Means k=2',       'Silhouette': sil,     'Purity': pur,     'Semi-sup F1': '-'},
    {'Method': 'K-Means k=4',       'Silhouette': sil4,    'Purity': '-',     'Semi-sup F1': '-'},
    {'Method': 'GMM n=2',           'Silhouette': sil_gmm, 'Purity': pur_gmm, 'Semi-sup F1': '-'},
    {'Method': 'Label Spreading',   'Silhouette': '-',     'Purity': '-',     'Semi-sup F1': round(lp_f1,4)},
])
print("=== Unsupervised / Semi-Supervised Results ===")
print(unsup_table.to_string(index=False))

=== Unsupervised / Semi-Supervised Results ===
         Method Silhouette  Purity Semi-sup F1
    K-Means k=2    0.08294  0.7543           -
    K-Means k=4   0.066979       -           -
        GMM n=2   0.069099  0.7543           -
Label Spreading          -       -      0.5045


## 12. Template-Based Question Generation with ML Ranker

**Step 1** — Extract candidate sentences using OHE keyword overlap with answer  
**Step 2** — Apply Wh-word templates  
**Step 3** — Rank with a trained SVM scorer

In [16]:
STOP_WORDS = set("a an the is was are were be been being have has had do does did "
                 "will would could should may might shall can must of in on at to "
                 "for with by from up about into through during before after i me "
                 "my we our you your he she it they them their its".split())

def sentence_tokenize(text):
    """Simple regex sentence splitter."""
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', str(text)) if len(s.strip()) > 10]

WH_TEMPLATES = [
    ('who',   r'\b(he|she|they|the \w+|a \w+)\b'),
    ('what',  r'\b(it|the \w+|a \w+|this|that)\b'),
    ('where', r'\b(in|at|on|near|by) (the )?\w+\b'),
    ('when',  r'\b(in|on|at|after|before|during) \d{4}\b'),
    ('why',   r'\bbecause\b'),
]

def apply_template(sentence, wh, pattern):
    """Replace first matching phrase with a blank for Wh-question template."""
    m = re.search(pattern, sentence, re.IGNORECASE)
    if m:
        return f"{wh.capitalize()} " + sentence[:m.start()] + "___" + sentence[m.end():]
    return None

def keyword_overlap(sent, answer_clean):
    s_words = {w for w in sent.lower().split() if w not in STOP_WORDS}
    a_words = {w for w in answer_clean.split()  if w not in STOP_WORDS}
    return len(s_words & a_words) / (len(s_words | a_words) + 1e-9)

def generate_candidate_questions(article, answer_clean, top_k=5):
    sents = sentence_tokenize(article)
    scored = [(s, keyword_overlap(s, answer_clean)) for s in sents]
    scored.sort(key=lambda x: x[1], reverse=True)
    candidates = []
    for sent, score in scored[:top_k]:
        for wh, pat in WH_TEMPLATES:
            q = apply_template(sent, wh, pat)
            if q:
                candidates.append({'question': q, 'source_sent': sent,
                                   'overlap': score, 'wh': wh})
    return candidates

# Demo
sample = train_df.iloc[0]
cands  = generate_candidate_questions(sample['article_clean'],
                                       sample['A_clean'])  # correct answer for demo
print(f"Generated {len(cands)} candidate questions for sample[0]:")
for c in cands[:3]:
    print(f"  [{c['wh'].upper()}] {c['question'][:120]}")

Generated 4 candidate questions for sample[0]:
  [WHO] Who research shows that humans switch from selfish to unselfish behavior when ___ are watched do you a picture of a set 
  [WHAT] What research shows ___ humans switch from selfish to unselfish behavior when they are watched do you a picture of a set
  [WHERE] Where research shows that humans switch from selfish to unselfish behavior when they are watched do you a picture of a s


In [17]:
# ── Train an SVM ranker on candidate question features ────────────────────────
# Build a small dataset: for each training sample, generate candidates;
# label candidates that have high overlap with the actual question as positive.

def question_fluency_features(cand_text, wh, overlap):
    """Simple feature vector for a candidate question."""
    return [
        overlap,
        len(cand_text.split()),          # question word count
        len(cand_text),                   # char length
        int(wh == 'what'),
        int(wh == 'who'),
        int(wh == 'where'),
        int(wh == 'when'),
        int(wh == 'why'),
        cand_text.count('_'),             # blank count
    ]

print("Building question ranker training set (sample 3000 rows) ...")
RANKER_SAMPLE = 3000
ranker_X, ranker_y = [], []

for _, row in train_df.head(RANKER_SAMPLE).iterrows():
    gold_q = row['question_clean']
    correct_opt = row['answer']
    answer_clean = row[f'{correct_opt}_clean']
    cands = generate_candidate_questions(row['article_clean'], answer_clean)
    for c in cands:
        # Label: 1 if generated question overlaps with gold question, else 0
        gold_words = set(gold_q.split())
        gen_words  = set(c['question'].lower().split())
        label = 1 if len(gold_words & gen_words) / (len(gold_words) + 1) > 0.3 else 0
        ranker_X.append(question_fluency_features(c['question'], c['wh'], c['overlap']))
        ranker_y.append(label)

ranker_X = np.array(ranker_X)
ranker_y = np.array(ranker_y)
print(f"Ranker dataset: {ranker_X.shape}, positive rate: {ranker_y.mean():.3f}")

from sklearn.svm import SVC
svm_ranker = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm_ranker.fit(ranker_X, ranker_y)
joblib.dump(svm_ranker, f'{MOUT}/question_ranker_svm.pkl')
print("Question ranker SVM saved.")

Building question ranker training set (sample 3000 rows) ...
Ranker dataset: (10266, 9), positive rate: 0.917
Question ranker SVM saved.


## 13. Ensemble — Soft Voting & Stacking

In [18]:
# ── Soft-vote ensemble using LR + NB (both work on sparse OHE) ───────────────
# SVM (LinearSVC) doesn't support predict_proba natively, so we wrap it
from sklearn.calibration import CalibratedClassifierCV

print("Calibrating SVM for probability output ...")
svm_cal = CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000, random_state=42),
                                  cv=3)
svm_cal.fit(X_train, y_train)

print("Building soft-vote ensemble (LR + NB + SVM_cal) ...")
ensemble = VotingClassifier(
    estimators=[('lr', lr), ('nb', nb), ('svm', svm_cal)],
    voting='soft',
    n_jobs=-1
)
# Note: lr and nb are already fitted; VotingClassifier will refit
# unless we pass pre-fitted estimators and set voting accordingly.
# For Kaggle, we refit to be clean:
ensemble.fit(X_train, y_train)

r_ens = evaluate_binary('Ensemble(Soft)', ensemble, X_val, y_val)
r_ens['mc_accuracy'] = mc_accuracy(ensemble, X_val, y_val_mc)
print(f"  Multi-class accuracy (val): {r_ens['mc_accuracy']:.4f}")
results.append(r_ens)
joblib.dump(ensemble, f'{MOUT}/ensemble_soft_vote.pkl')
print("  Saved.")

Calibrating SVM for probability output ...
Building soft-vote ensemble (LR + NB + SVM_cal) ...
[Ensemble(Soft)] val | Acc=0.7500  Macro-F1=0.4286  Prec=0.3750  Rec=0.5000  (0.3s)
  Multi-class accuracy (val): 0.3140
  Saved.


In [19]:
# ── Stacking ensemble ─────────────────────────────────────────────────────────
print("Building stacking ensemble ...")
stacking = StackingClassifier(
    estimators=[
        ('lr',  LogisticRegression(C=1.0, max_iter=500, solver='saga', random_state=42)),
        ('nb',  ComplementNB(alpha=0.5)),
    ],
    final_estimator=LogisticRegression(C=0.5, max_iter=500, random_state=42),
    cv=3,
    n_jobs=-1
)
stacking.fit(X_train, y_train)

r_stack = evaluate_binary('Stacking', stacking, X_val, y_val)
r_stack['mc_accuracy'] = mc_accuracy(stacking, X_val, y_val_mc)
print(f"  Multi-class accuracy (val): {r_stack['mc_accuracy']:.4f}")
results.append(r_stack)
joblib.dump(stacking, f'{MOUT}/ensemble_stacking.pkl')
print("  Saved.")

Building stacking ensemble ...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[Stacking] val | Acc=0.7500  Macro-F1=0.4286  Prec=0.3750  Rec=0.5000  (0.2s)
  Multi-class accuracy (val): 0.3042
  Saved.


## 14. Final Evaluation on Test Set

In [20]:
print("\n=== TEST SET EVALUATION ===")
best_models = [('LR', lr, X_test, y_test),
               ('NB', nb, X_test, y_test),
               ('Ensemble(Soft)', ensemble, X_test, y_test),
               ('Stacking', stacking, X_test, y_test)]

test_results = []
for name, model, Xt, yt in best_models:
    r = evaluate_binary(name, model, Xt, yt, split='test')
    r['mc_accuracy_test'] = mc_accuracy(model, Xt, y_test_mc)
    test_results.append(r)
    print(f"  MC-Acc(test): {r['mc_accuracy_test']:.4f}")

results_df = pd.DataFrame(results + test_results)
results_df.to_csv(f'{MOUT}/model_a_results.csv', index=False)
print("\nResults saved to model_a_results.csv")


=== TEST SET EVALUATION ===
[LR] test | Acc=0.7498  Macro-F1=0.4288  Prec=0.4821  Rec=0.5000  (0.6s)
  MC-Acc(test): 0.3026
[NB] test | Acc=0.5368  Macro-F1=0.4861  Prec=0.5048  Rec=0.5063  (0.1s)
  MC-Acc(test): 0.3373
[Ensemble(Soft)] test | Acc=0.7500  Macro-F1=0.4286  Prec=0.5417  Rec=0.5000  (0.3s)
  MC-Acc(test): 0.3124
[Stacking] test | Acc=0.7500  Macro-F1=0.4286  Prec=0.3750  Rec=0.5000  (0.1s)
  MC-Acc(test): 0.3056

Results saved to model_a_results.csv


## 15. Confusion Matrix & Performance Plots

In [21]:
# Confusion matrix for best model on test set
best = ensemble
y_pred_test = best.predict(X_test)
cm = confusion_matrix(y_test, y_pred_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix — Ensemble (Test)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

# Model comparison bar chart
val_res = [r for r in results if r.get('split') == 'val']
names  = [r['model']     for r in val_res]
accs   = [r['accuracy']  for r in val_res]
f1s    = [r['macro_f1']  for r in val_res]
x = np.arange(len(names))
w = 0.35
axes[1].bar(x - w/2, accs, w, label='Accuracy')
axes[1].bar(x + w/2, f1s,  w, label='Macro-F1')
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=20, ha='right')
axes[1].set_ylim(0, 1); axes[1].set_title('Model A — Val Set Comparison')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{MOUT}/model_a_performance.png', dpi=120)
plt.show()
print("Plot saved.")

Plot saved.


In [22]:
print("\n✅ Model A training complete.")
print("Saved models:")
for f in sorted(os.listdir(MOUT)):
    print(f"  {f}")


✅ Model A training complete.
Saved models:
  ensemble_soft_vote.pkl
  ensemble_stacking.pkl
  gmm_model.pkl
  kmeans_k2.pkl
  kmeans_k4.pkl
  label_prop_model.pkl
  lr_model.pkl
  model_a_performance.png
  model_a_results.csv
  nb_model.pkl
  question_ranker_svm.pkl
  rf_model.pkl
  svd_50.pkl
  svm_model.pkl
  xgb_model.json


In [23]:
import shutil
from IPython.display import FileLink

# 1. Define the target directory and the output zip name
folder_to_zip = '/kaggle/working/models'
output_filename = '/kaggle/working/all_models_backup' 

# 2. Create the zip archive
shutil.make_archive(output_filename, 'zip', folder_to_zip)
print(f"Successfully created {output_filename}.zip")

# 3. FIX: Generate the link using ONLY the relative filename
display(FileLink('all_models_backup.zip'))

Successfully created /kaggle/working/all_models_backup.zip


/kaggle/working/all_models_backup.zip